In [1]:
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

I0000 00:00:1785480008.218226  620292 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785480009.772428  620292 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785480016.714612  620292 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [6]:
train_data = pd.read_csv("../dataset/train_metadata.csv")
val_data = pd.read_csv("../dataset/val_metadata.csv")
test_data = pd.read_csv("../dataset/test_metadata.csv")

In [7]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [8]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [9]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [10]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

In [11]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.RandomZoom(0.1),

    tf.keras.layers.RandomContrast(0.1)

])

In [12]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [13]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [14]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [15]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

basic cnn with early stopping

In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models
num_classes = 7
basic_cnn = models.Sequential([
    layers.Input(shape=(224,224,3)),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128,activation="relu"),
    layers.Dense(num_classes,activation="softmax")
])


basic_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,908,295 (91.20 MB)

 Trainable params: 23,908,295 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
basic_cnn.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [14]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [16]:
history_basic = basic_cnn.fit(train_dataset,validation_data=val_dataset,epochs=10,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/10


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 307s 1s/step - accuracy: 0.4077 - loss: 1.7599 - val_accuracy: 0.1571 - val_loss: 1.8451
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.3312 - loss: 1.7309 - val_accuracy: 0.4933 - val_loss: 1.4701
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 339s 2s/step - accuracy: 0.4287 - loss: 1.5598 - val_accuracy: 0.3569 - val_loss: 1.7161
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 334s 1s/step - accuracy: 0.4502 - loss: 1.4518 - val_accuracy: 0.3429 - val_loss: 1.6122
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 343s 2s/step - accuracy: 0.4656 - loss: 1.3736 - val_accuracy: 0.4707 - val_loss: 1.3270
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 342s 2s/step - accuracy: 0.4312 - loss: 1.4055 - val_accuracy: 0.4274 - val_loss: 1.4214
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 338s 1s/step - accuracy: 0.4728 - loss: 1.3218 - val_accuracy: 0.5087 - val_loss: 1.1939
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 332s 1s/step - accuracy: 0.4596 - loss: 1.3421 - val_accuracy: 0.470

In [17]:
test_loss, test_accuracy = basic_cnn.evaluate(test_dataset)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 270ms/step - accuracy: 0.4917 - loss: 1.1798
Test Loss: 1.1798396110534668
Test Accuracy: 0.49168330430984497


In [19]:
basic_cnn.save("../models/basic_cnn_early_stopping.keras")

In [20]:
comparison_results = []

comparison_results.append({
    "Model": "Basic CNN + EarlyStopping",
    "Phase 4 Test Accuracy": 0.3180,
    "Phase 5 Test Accuracy": test_accuracy,
    "Phase 4 Test Loss": 1.5563,
    "Phase 5 Test Loss": test_loss
})

Basic cnn with learning rate

In [21]:
basic_cnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"]

)

In [22]:
history_basic = basic_cnn.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 319s 1s/step - accuracy: 0.5175 - loss: 1.2509 - val_accuracy: 0.5000 - val_loss: 1.1910
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 327s 1s/step - accuracy: 0.5140 - loss: 1.2512 - val_accuracy: 0.4960 - val_loss: 1.2066
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 341s 2s/step - accuracy: 0.5086 - loss: 1.2516 - val_accuracy: 0.4893 - val_loss: 1.2159
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 340s 2s/step - accuracy: 0.5090 - loss: 1.2194 - val_accuracy: 0.4933 - val_loss: 1.2067
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 314s 1s/step - accuracy: 0.5154 - loss: 1.2491 - val_accuracy: 0.4933 - val_loss: 1.2072


In [23]:
basic_cnn.save("../models/basic_cnn_learning_rate.keras")

In [24]:
test_loss, test_accuracy = basic_cnn.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 15s 326ms/step - accuracy: 0.4850 - loss: 1.2056
Test Loss: 1.2056121826171875
Test Accuracy: 0.485029935836792


In [25]:
learning_rate_results = []

learning_rate_results.append({

    "Model": "Basic CNN (LR=0.00001)",

    "Phase 4 LR": 0.0001,

    "Phase 5 LR": 0.00001,

    "Phase 4 Test Accuracy": 0.3180,

    "Phase 5 Test Accuracy": test_accuracy,

    "Phase 4 Test Loss": 1.5563,

    "Phase 5 Test Loss": test_loss

})

In [ ]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 16

train_dataset = (train_dataset.shuffle(len(train_data)).batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [14]:
import tensorflow as tf
from tensorflow.keras import layers, models

num_classes = 7

basic_cnn = models.Sequential([

    layers.Input(shape=(224,224,3)),

    layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D((2,2)),

    layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dense(
        num_classes,
        activation="softmax"
    )

])

basic_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,908,295 (91.20 MB)

 Trainable params: 23,908,295 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
basic_cnn.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights
)

Epoch 1/5


I0000 00:00:1785375506.854557   43001 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 1761 of 7010
I0000 00:00:1785375526.849759   43001 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 5139 of 7010
I0000 00:00:1785375537.504590   43001 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 6311 of 7010
I0000 00:00:1785375550.528817   43001 shuffle_dataset_op.cc:483] Shuffle buffer filled.


439/439 ━━━━━━━━━━━━━━━━━━━━ 401s 737ms/step - accuracy: 0.3853 - loss: 1.8592 - val_accuracy: 0.5233 - val_loss: 1.2230
Epoch 2/5


I0000 00:00:1785375905.131679   58365 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 729 of 7010
I0000 00:00:1785375925.110202   58365 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 3701 of 7010
I0000 00:00:1785375935.104026   58365 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 5304 of 7010
I0000 00:00:1785375945.113895   58365 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 6857 of 7010
I0000 00:00:1785375946.025729   58365 shuffle_dataset_op.cc:483] Shuffle buffer filled.


439/439 ━━━━━━━━━━━━━━━━━━━━ 372s 730ms/step - accuracy: 0.4435 - loss: 1.4862 - val_accuracy: 0.4674 - val_loss: 1.3058
Epoch 3/5


...

439/439 ━━━━━━━━━━━━━━━━━━━━ 401s 773ms/step - accuracy: 0.4815 - loss: 1.3553 - val_accuracy: 0.4514 - val_loss: 1.3395
Epoch 4/5


I0000 00:00:1785376677.814488   73863 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 1272 of 7010
I0000 00:00:1785376697.805948   73863 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 3475 of 7010
I0000 00:00:1785376707.838678   73863 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 4562 of 7010
I0000 00:00:1785376727.715658   73863 shuffle_dataset_op.cc:483] Shuffle buffer filled.


439/439 ━━━━━━━━━━━━━━━━━━━━ 406s 787ms/step - accuracy: 0.5004 - loss: 1.3102 - val_accuracy: 0.5719 - val_loss: 1.1018
Epoch 5/5


I0000 00:00:1785377083.370070   89729 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 992 of 7010
I0000 00:00:1785377103.354802   89729 shuffle_dataset_op.cc:453] ShuffleDatasetV3:21: Filling up shuffle buffer (this may take a while): 3866 of 7010
I0000 00:00:1785377122.874353   89729 shuffle_dataset_op.cc:483] Shuffle buffer filled.


439/439 ━━━━━━━━━━━━━━━━━━━━ 411s 824ms/step - accuracy: 0.5288 - loss: 1.2170 - val_accuracy: 0.5566 - val_loss: 1.1336


In [17]:
test_loss, test_accuracy = basic_cnn.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

94/94 ━━━━━━━━━━━━━━━━━━━━ 18s 185ms/step - accuracy: 0.5376 - loss: 1.1549
Test Loss: 1.154880404472351
Test Accuracy: 0.5375914573669434


In [18]:
batch_size_results = []

batch_size_results.append({

    "Model": "Basic CNN (Batch Size = 16)",

    "Phase 4 Batch Size": 32,

    "Phase 5 Batch Size": 16,

    "Phase 4 Test Accuracy": 0.3180,

    "Phase 5 Test Accuracy": test_accuracy,

    "Phase 4 Test Loss": 1.5563,

    "Phase 5 Test Loss": test_loss

})

In [19]:
print(batch_size_results)

[{'Model': 'Basic CNN (Batch Size = 16)', 'Phase 4 Batch Size': 32, 'Phase 5 Batch Size': 16, 'Phase 4 Test Accuracy': 0.318, 'Phase 5 Test Accuracy': 0.5375914573669434, 'Phase 4 Test Loss': 1.5563, 'Phase 5 Test Loss': 1.154880404472351}]


In [21]:
basic_cnn.save("../models/basic_cnn_batch16.keras")

In [12]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 64

train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models

num_classes = 7

basic_cnn = models.Sequential([

    layers.Input(shape=(224,224,3)),

    layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D((2,2)),

    layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dense(
        num_classes,
        activation="softmax"
    )

])

basic_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,908,295 (91.20 MB)

 Trainable params: 23,908,295 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
basic_cnn.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights
)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 325s 3s/step - accuracy: 0.3175 - loss: 2.0007 - val_accuracy: 0.4341 - val_loss: 1.7671
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 286s 3s/step - accuracy: 0.4272 - loss: 1.7486 - val_accuracy: 0.4188 - val_loss: 1.6145
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 332s 3s/step - accuracy: 0.4469 - loss: 1.6241 - val_accuracy: 0.6052 - val_loss: 1.1713
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 326s 3s/step - accuracy: 0.4722 - loss: 1.4996 - val_accuracy: 0.5240 - val_loss: 1.2219
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 341s 3s/step - accuracy: 0.4996 - loss: 1.3893 - val_accuracy: 0.4740 - val_loss: 1.4704


In [15]:
basic_cnn.save("../models/basic_cnn_batch64.keras")

In [16]:
test_loss, test_accuracy = basic_cnn.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

24/24 ━━━━━━━━━━━━━━━━━━━━ 18s 707ms/step - accuracy: 0.4617 - loss: 1.4920
Test Loss: 1.4920055866241455
Test Accuracy: 0.46174317598342896


In [17]:
batch_size_results = []

batch_size_results.append({

    "Model": "Basic CNN (Batch Size = 64)",

    "Phase 4 Batch Size": 32,

    "Phase 5 Batch Size": 16,

    "Phase 4 Test Accuracy": 0.3180,

    "Phase 5 Test Accuracy": test_accuracy,

    "Phase 4 Test Loss": 1.5563,

    "Phase 5 Test Loss": test_loss

})

In [20]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)

BATCH_SIZE = 32

train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

Optimizer Chnge

In [21]:
basic_cnn.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=10,

    class_weight=class_weights
)

Epoch 1/10


220/220 ━━━━━━━━━━━━━━━━━━━━ 273s 1s/step - accuracy: 0.4981 - loss: 1.2884 - val_accuracy: 0.5200 - val_loss: 1.2557
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 261s 1s/step - accuracy: 0.5160 - loss: 1.2787 - val_accuracy: 0.5379 - val_loss: 1.2079
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 269s 1s/step - accuracy: 0.5184 - loss: 1.2706 - val_accuracy: 0.5326 - val_loss: 1.2479
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 269s 1s/step - accuracy: 0.5213 - loss: 1.2632 - val_accuracy: 0.5513 - val_loss: 1.1787
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 298s 1s/step - accuracy: 0.5298 - loss: 1.2632 - val_accuracy: 0.5273 - val_loss: 1.2579
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 333s 1s/step - accuracy: 0.5295 - loss: 1.2512 - val_accuracy: 0.4800 - val_loss: 1.3047
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.5243 - loss: 1.2521 - val_accuracy: 0.5546 - val_loss: 1.1397
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.5355 - loss: 1.2466 - val_accuracy: 0.470

In [22]:
basic_cnn.save("../models/basic_cnn_sgd.keras")

In [23]:
test_loss, test_accuracy = basic_cnn.evaluate(test_dataset)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 365ms/step - accuracy: 0.5110 - loss: 1.2375
Test Loss: 1.2375216484069824
Test Accuracy: 0.5109780430793762


In [24]:
basic_cnn.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

history_basic = basic_cnn.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 338s 1s/step - accuracy: 0.5136 - loss: 1.2981 - val_accuracy: 0.5533 - val_loss: 1.0648
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 303s 1s/step - accuracy: 0.5308 - loss: 1.2697 - val_accuracy: 0.5213 - val_loss: 1.2344
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 286s 1s/step - accuracy: 0.5351 - loss: 1.2328 - val_accuracy: 0.4720 - val_loss: 1.2943
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 284s 1s/step - accuracy: 0.5452 - loss: 1.2049 - val_accuracy: 0.4953 - val_loss: 1.4490
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 296s 1s/step - accuracy: 0.5427 - loss: 1.2014 - val_accuracy: 0.5546 - val_loss: 1.0750


In [ ]:
basic_cnn.save("../models/basic_cnn_rmsprop.keras")

In [ ]:
test_loss, test_accuracy = basic_cnn.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [12]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

num_classes = 7

def build_model(hp):

    model = models.Sequential([

        layers.Input(shape=(224,224,3)),

        layers.Conv2D(

            filters=hp.Choice(
                "filters_1",
                [32,64]
            ),

            kernel_size=(3,3),

            activation="relu"
        ),

        layers.MaxPooling2D((2,2)),


        layers.Conv2D(

            filters=hp.Choice(
                "filters_2",
                [64,128]
            ),

            kernel_size=(3,3),

            activation="relu"
        ),

        layers.MaxPooling2D((2,2)),


        layers.Flatten(),


        layers.Dense(

            units=hp.Choice(
                "dense_units",
                [64,128,256]
            ),

            activation="relu"
        ),


        layers.Dense(

            num_classes,

            activation="softmax"
        )

    ])


    learning_rate = hp.Choice(

        "learning_rate",

        [1e-2,1e-3,1e-4]
    )


    optimizer = hp.Choice(

        "optimizer",

        ["adam","sgd","rmsprop"]
    )


    if optimizer == "adam":

        opt = Adam(
            learning_rate=learning_rate
        )

    elif optimizer == "sgd":

        opt = SGD(
            learning_rate=learning_rate
        )

    else:

        opt = RMSprop(
            learning_rate=learning_rate
        )


    model.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]
    )

    return model

In [15]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=3,

    overwrite=True,

    directory="tuner",

    project_name="basic_cnn"
)

In [16]:
tuner.search(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights
)

Trial 3 Complete [00h 13m 19s]

Best val_accuracy So Far: 0.6864181160926819
Total elapsed time: 02h 25m 59s


best_bn_cnn = tuner.get_best_models(1)[0]

In [17]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'filters_1': 64, 'filters_2': 64, 'dense_units': 128, 'learning_rate': 0.0001, 'optimizer': 'rmsprop'}


In [15]:

num_classes = 7
final_cnn = models.Sequential([
    layers.Input(shape=(224,224,3)),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128,activation="relu"),
    layers.Dense(num_classes,activation="softmax")
])


final_cnn.summary()

W0000 00:00:1785466962.088709  262997 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.
W0000 00:00:1785466962.188353  262997 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.
W0000 00:00:1785466962.217940  262997 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,927,623 (91.28 MB)

 Trainable params: 23,927,623 (91.28 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
final_cnn.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

history_cnn_final = final_cnn.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights
)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
W0000 00:00:1785466966.760038  262997 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.
W0000 00:00:1785466967.789152  263273 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.


220/220 ━━━━━━━━━━━━━━━━━━━━ 705s 3s/step - accuracy: 0.3723 - loss: 1.9174 - val_accuracy: 0.3495 - val_loss: 1.9554
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 870s 4s/step - accuracy: 0.4454 - loss: 1.7066 - val_accuracy: 0.3702 - val_loss: 1.8109
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 848s 4s/step - accuracy: 0.4592 - loss: 1.5712 - val_accuracy: 0.5013 - val_loss: 1.3654
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 715s 3s/step - accuracy: 0.4777 - loss: 1.4661 - val_accuracy: 0.3968 - val_loss: 1.6594
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 613s 3s/step - accuracy: 0.4929 - loss: 1.4059 - val_accuracy: 0.5013 - val_loss: 1.3174


In [19]:
from tensorflow.keras.models import load_model
basic_load=load_model( "../models/final_cnn.keras")

In [20]:
train_loss, train_accuracy = basic_load.evaluate(train_dataset)
val_loss, val_accuracy = basic_load.evaluate(val_dataset)
test_loss, test_accuracy = basic_load.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 123s 543ms/step - accuracy: 0.4923 - loss: 1.2959
47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 450ms/step - accuracy: 0.5013 - loss: 1.3174
47/47 ━━━━━━━━━━━━━━━━━━━━ 22s 465ms/step - accuracy: 0.4744 - loss: 1.3410


final_cnn.save("../models/final_cnn.keras")

In [3]:
from tensorflow.keras.models import load_model
best_model_cnn=load_model("../models/basic_cnn_batch16.keras")

E0000 00:00:1785480073.770126  620292 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [24]:
train_loss, train_accuracy = best_model_cnn.evaluate(train_dataset)
val_loss, val_accuracy = best_model_cnn.evaluate(val_dataset)
test_loss, test_accuracy = best_model_cnn.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 97s 410ms/step - accuracy: 0.5505 - loss: 1.1101
47/47 ━━━━━━━━━━━━━━━━━━━━ 15s 307ms/step - accuracy: 0.5566 - loss: 1.1336
47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 296ms/step - accuracy: 0.5376 - loss: 1.1549
